# KPI and Licensing Dashboard

Downloads the appliance licensing archive, calculates discovery KPIs and estimates resource-unit consumption.

## Setup

Select the appliance and configure optional KPI targets. Results remain in memory unless `WRITE_OUTPUT` is enabled.

In [ ]:
import io
import math
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display
from tideway import notebooks as tw_nb

APPLIANCE_NAME = None
APPLIANCE_INDEX = 0
WRITE_OUTPUT = False
OUTPUT_BASE_DIR = None
DISPLAY_ROW_LIMIT = 50
LICENSE_THRESHOLD_OVERRIDE = None

KPI_TARGETS = {
    'Servers': None,
    'Windows Desktops': None,
    'Network Devices': None,
    'Software Instances': None,
    'Virtual Machines': None,
    'Storage Ports': None,
    'Containers/Pods': None,
    'Cloud/PaaS Resources': None,
}
CUSTOM_KPI_FILTERS = []

display(HTML('''<style>
.jp-OutputArea-output, .output_area, .dataframe { width: 100% !important; }
.dataframe { table-layout: auto !important; }
</style>'''))

def display_table(frame, limit=DISPLAY_ROW_LIMIT):
    display(frame.head(limit).style.set_table_attributes('style="width:100%"'))

repo_root = tw_nb.find_repo_root(Path.cwd())
config = tw_nb.load_config()
selected_config = tw_nb.selected_appliance_config(config, APPLIANCE_NAME, APPLIANCE_INDEX)
tw = tw_nb.appliance_from_config(appliance_name=APPLIANCE_NAME, appliance_index=APPLIANCE_INDEX)
search_api = tw.data()
admin_api = tw.admin()
output_dir = tw_nb.output_dir_for(tw.target, OUTPUT_BASE_DIR)

configured_threshold = selected_config.get('license_threshold', config.get('license_threshold'))
license_threshold = LICENSE_THRESHOLD_OVERRIDE if LICENSE_THRESHOLD_OVERRIDE is not None else configured_threshold
license_threshold = int(license_threshold) if license_threshold not in (None, '', 'None') else None

print('Target            :', tw.target)
print('Output directory  :', output_dir)
print('Write output      :', WRITE_OUTPUT)
print('License threshold :', license_threshold)

## Licensing Archive

The licensing endpoint returns a ZIP archive. Each contained CSV is displayed separately.

In [ ]:
licensing_response = admin_api.get_admin_licensing('csv')
if not getattr(licensing_response, 'ok', False):
    status = getattr(licensing_response, 'status_code', 'unknown')
    raise RuntimeError(f'Licensing API request failed with status {status}')

archive_bytes = licensing_response.content
licensing_tables = {}
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for filename in archive.namelist():
        if filename.lower().endswith('.csv'):
            licensing_tables[filename] = pd.read_csv(io.BytesIO(archive.read(filename)))

if not licensing_tables:
    raise ValueError('The licensing archive did not contain any CSV reports.')

licensing_summary = []
for filename, frame in licensing_tables.items():
    identifiers = frame['Identifier'].nunique() if 'Identifier' in frame.columns else len(frame)
    licensing_summary.append({
        'Report': filename,
        'Rows': len(frame),
        'Unique identifiers': int(identifiers),
        'First seen': frame['First seen'].min() if 'First seen' in frame.columns and not frame.empty else None,
        'Last seen': frame['Last seen'].max() if 'Last seen' in frame.columns and not frame.empty else None,
    })
    print(filename)
    display_table(frame)

licensing_summary_df = pd.DataFrame(licensing_summary)
display_table(licensing_summary_df)

if WRITE_OUTPUT:
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'license.zip').write_bytes(archive_bytes)
    for filename, frame in licensing_tables.items():
        frame.to_csv(output_dir / f'licensing_{Path(filename).name}', index=False)
    licensing_summary_df.to_csv(output_dir / 'licensing_summary.csv', index=False)
    print('Saved licensing files to', output_dir)

## Discovery Counts

In [ ]:
count_failures = []

def count_query(node, where=None):
    query = f'search {node}'
    if where:
        query += f' where {where}'
    return query + ' process with unique(0)'

def get_count(node, where=None):
    query = count_query(node, where)
    try:
        result = search_api.search({'query': query}, format='object')
        if hasattr(result, 'ok'):
            status = getattr(result, 'status_code', 'unknown')
            raise RuntimeError(f'API request failed with status {status}')
        if not isinstance(result, list):
            raise TypeError(f'Unexpected result type: {type(result).__name__}')
        return len(result)
    except Exception as exc:
        count_failures.append({'Node': node, 'Where': where, 'Error': str(exc)})
        return 0

def first_available_count(nodes, where=None):
    for node in nodes:
        before = len(count_failures)
        count = get_count(node, where)
        if len(count_failures) == before:
            return count
    return 0

top_nodes = [
    'Host', 'NetworkDevice', 'Printer', 'StorageSystem', 'StorageDevice',
    'SoftwareInstance', 'VirtualMachine', 'SNMPManagedDevice', 'ManagementController',
]
counts = [{'Node': node, 'Count': get_count(node)} for node in top_nodes]
storage_ports = first_available_count(['StoragePort'])
containers = first_available_count(['SoftwarePod', 'SoftwareContainer'])
cloud_resources = first_available_count(['CloudResource', 'CloudService'])
counts.extend([
    {'Node': 'StoragePort', 'Count': storage_ports},
    {'Node': 'Containers/Pods', 'Count': containers},
    {'Node': 'Cloud/PaaS', 'Count': cloud_resources},
])
counts_df = pd.DataFrame(counts).sort_values('Node').reset_index(drop=True)
display_table(counts_df)

if WRITE_OUTPUT:
    output_dir.mkdir(parents=True, exist_ok=True)
    counts_df.to_csv(output_dir / 'node_counts.csv', index=False)

## KPI Targets

In [ ]:
server_where = (
    "not (host_type has subword 'desktop' or host_type has subword 'client' or host_type has subword 'laptop') "
    "and not (cloud and nodecount(traverse ContainedHost:HostContainment:HostContainer:VirtualMachine where cloud)) "
    "and nodecount(traverse InferredElement:Inference::DiscoveryAccess)"
)
desktop_where = "host_type has subword 'desktop' or host_type has subword 'laptop'"
cloud_where = "__resource_unit = 'cloud'"

discovered = {
    'Servers': get_count('Host', server_where),
    'Windows Desktops': get_count('Host', desktop_where),
    'Network Devices': get_count('NetworkDevice'),
    'Software Instances': get_count('SoftwareInstance'),
    'Virtual Machines': get_count('VirtualMachine'),
    'Storage Ports': storage_ports,
    'Containers/Pods': containers,
    'Cloud/PaaS Resources': get_count('SoftwareInstance,VirtualMachine', cloud_where),
}

kpi_rows = []
for name, target in KPI_TARGETS.items():
    found = int(discovered.get(name, 0))
    target_value = int(target) if target not in (None, '') else None
    achieved = round(100.0 * found / target_value, 2) if target_value and target_value > 0 else None
    shortfall = max(0, target_value - found) if target_value is not None else None
    kpi_rows.append({
        'KPI': name, 'Target': target_value, 'Discovered': found,
        'Achieved %': achieved, 'Shortfall': shortfall,
    })

for specification in CUSTOM_KPI_FILTERS:
    name = specification.get('name') or specification.get('node', 'Custom KPI')
    found = get_count(specification.get('node', 'Host'), specification.get('where'))
    target = specification.get('target')
    target_value = int(target) if target not in (None, '') else None
    achieved = round(100.0 * found / target_value, 2) if target_value and target_value > 0 else None
    shortfall = max(0, target_value - found) if target_value is not None else None
    kpi_rows.append({
        'KPI': name, 'Target': target_value, 'Discovered': found,
        'Achieved %': achieved, 'Shortfall': shortfall,
    })

kpi_df = pd.DataFrame(kpi_rows)
display_table(kpi_df)
if WRITE_OUTPUT:
    kpi_df.to_csv(output_dir / 'kpi_targets.csv', index=False)

## Resource Units

In [ ]:
ru_categories = [
    ('Servers', discovered['Servers'], 1),
    ('Network Devices', discovered['Network Devices'], 5),
    ('Storage Ports', discovered['Storage Ports'], 5),
    ('Clients (Desktops/Laptops)', discovered['Windows Desktops'], 5),
    ('Containers/Pods', discovered['Containers/Pods'], 20),
    ('Cloud/PaaS Resources', discovered['Cloud/PaaS Resources'], 5),
]
ru_rows = [
    {'Category': name, 'Count': int(count), 'Units per RU': divisor, 'RUs': math.ceil(count / divisor)}
    for name, count, divisor in ru_categories
]
ru_df = pd.DataFrame(ru_rows)
ru_total = int(ru_df['RUs'].sum())
ru_df.loc[len(ru_df)] = {
    'Category': 'TOTAL', 'Count': int(ru_df['Count'].sum()),
    'Units per RU': None, 'RUs': ru_total,
}
display_table(ru_df)
print('Estimated RUs    :', ru_total)
if license_threshold is not None:
    print('License threshold:', license_threshold)
    print('Remaining RUs    :', license_threshold - ru_total)

if WRITE_OUTPUT:
    ru_df.to_csv(output_dir / 'ru_breakdown.csv', index=False)

## Charts

In [ ]:
target_plot = kpi_df.dropna(subset=['Target'])
if not target_plot.empty:
    figure, axis = plt.subplots(figsize=(12, 5))
    positions = list(range(len(target_plot)))
    axis.bar([position - 0.2 for position in positions], target_plot['Target'], width=0.4, label='Target')
    axis.bar([position + 0.2 for position in positions], target_plot['Discovered'], width=0.4, label='Discovered')
    axis.set_xticks(positions, target_plot['KPI'], rotation=45, ha='right')
    axis.set_ylabel('Count')
    axis.set_title('KPI Targets vs Discovered')
    axis.legend()
    figure.tight_layout()
    if WRITE_OUTPUT:
        figure.savefig(output_dir / 'kpi_targets.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No KPI targets configured; target chart skipped.')

ru_plot = ru_df[ru_df['Category'] != 'TOTAL']
figure, axis = plt.subplots(figsize=(10, 5))
bottom = 0
for _, row in ru_plot.iterrows():
    axis.bar('Total', row['RUs'], bottom=bottom, label=row['Category'])
    bottom += row['RUs']
if license_threshold is not None:
    axis.axhline(license_threshold, linestyle='--', color='black', label=f'License threshold ({license_threshold})')
axis.set_ylabel('Resource Units')
axis.set_title('Estimated License RU Consumption')
axis.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
figure.tight_layout()
if WRITE_OUTPUT:
    figure.savefig(output_dir / 'ru_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## Query Status

In [ ]:
if count_failures:
    failures_df = pd.DataFrame(count_failures)
    print(f'{len(failures_df)} count queries failed; affected counts are shown as zero.')
    display_table(failures_df)
else:
    print('All count queries completed successfully.')